In [ ]:
import os
import random
from dotenv import load_dotenv
from huggingface_hub import login
from datasets import load_dataset, Dataset, DatasetDict
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import numpy as np
import pickle

In [ ]:
import package.global_vars

In [ ]:
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
from package.items import Item
from package.loaders import ItemLoader

In [ ]:
%matplotlib inline


In [ ]:
dataset_names =     "Appliances"
items = []
#for d in dataset_names:
loader = ItemLoader(dataset_names)
items.extend(loader.load())

In [ ]:
sample = items
sizes = [len(item.prompt) for item in items]
prices = [item.price for item in sample]

plt.figure(figsize=(15,8))
plt.scatter(sizes, prices, s=0.2, color = "red")

plt.xlabel('Sizes')
plt.ylabel('Prices')
plt.show()

In [ ]:
def report(item):
    prompt = item.prompt
    tokens = Item.tokenizer.encode(item.prompt)
    print(prompt)
    print(item.test_prompt())
    print(tokens[-10:])
    print(Item.tokenizer.batch_decode(tokens[-10:]))

In [ ]:
report(sample[50])

In [ ]:
random.seed(42)
random.shuffle(sample)
train = sample[:25000]
test = sample[25000:27000]
print(f"Divided into a training set of {len(train):,} items and test set of {len(test):,} items")
print(report(train[0]))

In [ ]:
train_prompts = [item.prompt for item in train]
train_prices = [item.price for item in train]
test_prompts = [item.prompt for item in test]
test_prices = [item.price for item in test]

In [ ]:
train_dataset = Dataset.from_dict({"text":train_prompts, "prices":train_prices})
test_dataset = Dataset.from_dict({"text": test_prompts, "prices":test_prices})
dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
    })

In [ ]:
DATASET_NAME= "ananthak86/lite-data"
dataset.push_to_hub(DATASET_NAME)

In [ ]:
with open('train_lite.pkl', 'wb') as file:
    pickle.dump(train, file)

with open('test_lite.pkl', 'wb') as file:
    pickle.dump(test, file)